In [ ]:
from pathlib import Path
import os

import napari
import itertools
import math

# matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec
from matplotlib import rc

import numpy as np

import operator

# infer-subc
from infer_subc.utils.stats import create_overlap
from infer_subc.core.file_io import (read_czi_image,
                                     import_inferred_organelle,
                                     list_image_files)
from infer_subc.core.img import apply_mask

In [ ]:
## Define the path to the directory that contains the input image folder.
data_root_path = Path(os.path.expanduser("~")) / "Documents/Python Scripts/Infer-subc-2D/ncont vis/neuron"

## Specify which subfolder that contains the input data and what the file type is. Ex) ".czi" or ".tiff"
in_data_path = data_root_path / "raw"
raw_img_type = ".tiff"

## Specify which subfolder contains the segmentation outputs and their file type
seg_data_path = data_root_path / "seg"
seg_img_type = ".tiff"

## Specify the name of the output folder where quantification results will be saved
out_data_path = data_root_path / "quant"

## Specify which file you'd like to segment from the img_file_list
test_img_n = 0

## Specify the suffixes on the organelle segmentation files (i.e., the stuff following the "-")
org_names = ["lyso", "mito", "golgi", "perox", "ER", "LD"]

## Specify the suffixes on the region segmentation files (i.e., the stuff following the "-")
region_names = ["nuc", "cell"]

## Specify the organelles present in each channel of the raw image 
                ## syntax is {"organelle name": "channel number",...}
channel_orgs = {"lyso": 4,
                "mito": 3,
                "golgi": 2,
                "perox": 1,
                "ER": 0,
                "LD": 6,
                "nuc": 7,
                "PM": 5}

## Specify the suffix on the region segmentation file pertaining to the cell mask
mask = "cell"

## Specify the "splitter" that will be used between groupings of organelle names (i.e. lysoXmito)
splitter = "X"

In [ ]:
if not Path.exists(out_data_path):
    Path.mkdir(out_data_path)
    print(f"making {out_data_path}")

raw_file_list = list_image_files(in_data_path, raw_img_type)
seg_file_list = list_image_files(seg_data_path, seg_img_type)
# pd.set_option('display.max_colwidth', None)
# pd.DataFrame({"Image Name":img_file_list})

In [ ]:
raw_img_name = raw_file_list[test_img_n]

raw_img_data, raw_meta_dict = read_czi_image(raw_img_name)

channel_names = raw_meta_dict['name']
img = raw_meta_dict['metadata']['aicsimage']
scale = raw_meta_dict['scale']
channel_axis = raw_meta_dict['channel_axis']

In [ ]:
org_segs = [import_inferred_organelle(org, raw_meta_dict, seg_data_path, seg_img_type) for org in org_names]
region_segs = [import_inferred_organelle(reg, raw_meta_dict, seg_data_path, seg_img_type) for reg in region_names]

In [ ]:
mask_img = region_segs[region_names.index("cell")]

In [ ]:
organelle_segs = {}                                                     
for idx, name in enumerate(org_names):                                  
    if name == 'ER':                                                    
        organelle_segs[name]=apply_mask((org_segs[idx]>0).astype(np.uint16), mask_img)    
    else:                                                       
        organelle_segs[name]=org_segs[idx]                      

In [ ]:
def crop_to_square(img):
    yy,xx,cc = img.shape
    if yy > xx:
        bounding = (xx, xx, cc)
    elif xx > yy:
        bounding = (yy, yy, cc)
    else:
        return img
    start = tuple(map(lambda a, da: a//2-da//2, img.shape, bounding))
    end = tuple(map(operator.add, start, bounding))
    slices = tuple(map(slice, start, end))
    return img[slices]

In [ ]:
def get_bounded_font_size(width: float, height:float, text: str, base_size:float, fig, weight = None, style='Arial'):
    rc('font',family=style)
    r = fig.canvas.get_renderer()
    fsr = list(range(base_size+1))
    while (len(fsr) > 2):
        idx = len(fsr)//2
        fs = fsr[idx]
        t = plt.text(0.5, 0.5,text, fontsize=fs, fontweight=weight)
        if width <= t.get_window_extent(renderer=r).width:
            fsr = fsr[:idx+1]
        elif height <= t.get_window_extent(renderer=r).height:
            fsr = fsr[:idx+1]
        elif width >= t.get_window_extent(renderer=r).width:
            fsr = fsr[idx:]
        elif height >= t.get_window_extent(renderer=r).height:
            fsr = fsr[idx:]
        t.remove()
    if len(fsr) == 2:
        return (fsr[0]+fsr[1])/2
    else:
        return fs

In [ ]:
def plot_n_overlaps(orgs: str, 
                    splitter: str, 
                    organelle_segs: dict, 
                    scale: any, 
                    padding: float=0.05, 
                    close_viewer: bool=False, 
                    view='2D', 
                    colorblind:bool=False,
                    fontstyle:str='Arial',
                    dpi:int=300):
    """
    Plots nth order overlaps of organelles in a grid format. Only works for up to 6 organelles.

    Parameters
    ----------

    Returns
    -------
    plt
        Figure displaying the nth dimensional image alongside the lower order interactions

    """
    # Assigning Font Constant
    rc('font',family=fontstyle)

    # Assigning DPI constnat
    plt.rcParams['figure.dpi']=dpi

    # Assigning Color Constants
    if colorblind:
        ORG_A_COL = "#FFAE00"
        ORG_B_COL = "#00A0FC"
        ORG_C_COL = "#FF93CE"
        ORG_D_COL = "#01CE97"
        ORG_E_COL = '#F0E442'
        ORG_F_COL = "#FF6F01"
    else:
        ORG_A_COL = "#00D2D2"
        ORG_B_COL = "#D200D2"
        ORG_C_COL = "#D2D200"
        ORG_D_COL = "#00F900"
        ORG_E_COL = "#F90000"
        ORG_F_COL = "#0000F9"
    HO_MERGE = '#FFFFFF'
    LO_MERGE = '#999999'
    ORG_COLORS = [ORG_A_COL, ORG_B_COL, ORG_C_COL, ORG_D_COL, ORG_E_COL, ORG_F_COL]

    # Initializing the viewer
    viewer = napari.Viewer()
    viewer.add_labels(np.zeros_like(organelle_segs[list(organelle_segs.keys())[0]]), visible = False, scale=scale)

    plt_org = {}            # Dictionary of the images to plot
    axes = {}               # Dictionary of the axes to plot on
    title_axes = {}         # Dictionary of the title axes
    counts = []             # List of number of plots per section
    y_titles = {}           # Dictionary of titles used vertically
    x_titles = {}           # Dictionary of titles used horizontally

    if view == '3D':
        op = 0.33
        viewer.dims.ndisplay = 3
    elif view == '2D':
        op = 1.0
    elif type(view) == int:
        op = 1.0
        viewer.dims.set_point(0, view)
    else:
        raise ValueError("view must be either '2D' or '3D'")

    # Generate all possible combinations of the organelles
    all_pos =[]
    for n in list(map(lambda x:x+2, (range(len(orgs.split(splitter))-1)))):
        all_pos += itertools.combinations(orgs.split(splitter), n)
        counts.append(math.comb(len(orgs.split(splitter)), n))
    possib = [splitter.join(inter) for inter in all_pos]

    # Determine number of each order of LO overlaps
    counts = counts[:-1] + [len(orgs.split(splitter))+1]

    # Determine the minimum gridspec area
    grid_width = math.lcm(*counts)
    ti_spec = 0

    # Determine title area size, and adjust grid_width as needed to get a non-zero number
    while ti_spec == 0:
        min_spec = grid_width//max(counts)
        ti_spec = min_spec//(len(orgs.split(splitter))-1)
        if ti_spec == 0:
            grid_width *= 2
    grid_height = sum((grid_width//h) for h in counts[:-1]) + ((grid_width//(len(orgs.split(splitter))+1))*(len(orgs.split(splitter)))) # height of large plot
    # Adds organelles and their combinations to the viewer, and exports the images to the plt_org dictionary
    for i, org in enumerate(orgs.split(splitter) + possib):
        if org in organelle_segs.keys():
            viewer.add_labels(organelle_segs[org]>0, name=f"{org}_LO", scale=scale, opacity=op, colormap={None:None,1:ORG_COLORS[i]}, visible=True)
            plt_org[org] = crop_to_square(viewer.screenshot(canvas_only=True))
        else:
            viewer.add_labels((create_overlap(org, organelle_segs, splitter)>0), name=f"{org}_LO", scale=scale, opacity=1.0, colormap={None:None,1:LO_MERGE}, visible=False)
            viewer.add_labels((create_overlap(org, organelle_segs, splitter)>0), name=f"{org}_HO", scale=scale, opacity=1.0, colormap={None:None,1:HO_MERGE}, visible=True)
            plt_org[f"{org}_ol"] = crop_to_square(viewer.screenshot(canvas_only=True))
            for sub_org in (org.split(splitter) + [inter for inter in possib if (all(sub in org for sub in inter.split(splitter)) and inter != org)]):
                viewer.layers[f"{sub_org}_LO"].visible = True
            plt_org[org] = crop_to_square(viewer.screenshot(canvas_only=True))
        for layer in viewer.layers:
            layer.visible = False
    
    if close_viewer:
        viewer.close()

    fig = plt.figure(figsize=((2*grid_width),(grid_height+(ti_spec*len(counts))))) 

    # Initialize the grid spec
    if grid_width > grid_height:
        fig = plt.figure(figsize=((2*grid_width/(grid_height+(ti_spec*len(counts))))*36,36)) 
        gs = fig.add_gridspec(int(grid_height+(ti_spec*len(counts))), int(2*grid_width), wspace=(padding*2*(grid_width/(grid_height+(ti_spec*len(counts))))), hspace=padding)
    elif grid_height > grid_width:
        fig = plt.figure(figsize=((2)*36,((grid_height+(ti_spec*len(counts)))/(grid_width))*36)) 
        gs = fig.add_gridspec(int(grid_height+(ti_spec*len(counts))), int(2*grid_width), wspace=padding, hspace=(padding*2*((grid_height+(ti_spec*len(counts)))/grid_width)))  
    else:
        fig = plt.figure(figsize=((2)*36,36)) 
        gs = fig.add_gridspec(int(grid_height+(ti_spec*len(counts))), int(2*grid_width), wspace=padding, hspace=padding)

    # Merge Plots
    for n in list(map(lambda x:x+2, (range(len(orgs.split(splitter))-1)))): # n = overlap order number
        if n == (len(orgs.split(splitter))):
            # Full Merge
            axes[orgs] = fig.add_subplot(gs[int(grid_height-(len(orgs.split(splitter))*(grid_width/(len(orgs.split(splitter))+1)))+(ti_spec*(n-1))):,
                                            int(grid_width/(len(orgs.split(splitter))+1)):int(grid_width)])
            axes[f"{orgs}_ol"] = fig.add_subplot(gs[int(grid_height - (len(orgs.split(splitter))*(grid_width/(len(orgs.split(splitter))+1)))+(ti_spec*(n-1))):,
                                                    int(grid_width):int((2*grid_width)-(grid_width/(len(orgs.split(splitter))+1)))])
            x_titles[orgs] = orgs
            title_axes[orgs] = fig.add_subplot(gs[int(grid_height-(len(orgs.split(splitter))*(grid_width/(len(orgs.split(splitter))+1)))+(ti_spec*(n-2))):int(grid_height-(len(orgs.split(splitter))*(grid_width/(len(orgs.split(splitter))+1)))+(ti_spec*(n-1))),
                                                  int(grid_width/(len(orgs.split(splitter))+1)):int((2*grid_width)-(grid_width/(len(orgs.split(splitter))+1)))])
            # Single organelles
            y = int(sum((grid_width/counts[j-1]) for j in range(1, len(orgs.split(splitter))))) + (ti_spec*(n-1))
            x_titles["orgs"] = "Organelles"
            for i, org in enumerate(orgs.split(splitter)):
                if i == 0:
                    title_axes["orgs"] = fig.add_subplot(gs[int((y-(grid_width/(len(orgs.split(splitter))+1))+(i*(grid_width/(len(orgs.split(splitter))+1))))-ti_spec):int((y-(grid_width/(len(orgs.split(splitter))+1))+(i*(grid_width/(len(orgs.split(splitter))+1))))),
                                                            int(0):int(grid_width/(len(orgs.split(splitter))+1))])
                y_titles[org] = org
                axes[org] = fig.add_subplot(gs[int((y-(grid_width/(len(orgs.split(splitter))+1))+(i*(grid_width/(len(orgs.split(splitter))+1))))):int(y+((i)*(grid_width/(len(orgs.split(splitter))+1)))),
                                               int(0):int(grid_width/(len(orgs.split(splitter))+1))])
        else:
            # Lower order interactions
            y = int(sum((grid_width/counts[j-1]) for j in range(1, n))) + (ti_spec*(n-1))
            for i, org in enumerate([splitter.join(inter) for inter in itertools.combinations(orgs.split(splitter), n)]):   # i = overlap image number within the order
                if i == 0:
                    y_titles[org] = f"Interaction Order: {n}"
                axes[org] = fig.add_subplot(gs[int(y - (grid_width/counts[n-2])):y,
                                            int(((2*i)*(grid_width/counts[n-2]))):int(((2*i*(grid_width/counts[n-2]))+(grid_width/counts[n-2])))])
                axes[f"{org}_ol"] = fig.add_subplot(gs[int(y - (grid_width/counts[n-2])):y,
                                                       int(((2*i)*(grid_width/counts[n-2]))+(grid_width/counts[n-2])):int(((2*i)*((grid_width/counts[n-2]))+(2*(grid_width/counts[n-2]))))])
                # Titles
                x_titles[org] = org
                title_axes[org] = fig.add_subplot(gs[int((y - (grid_width/counts[n-2])) - (ti_spec)):int(y - (grid_width/counts[n-2])),
                                                     int(((2*i)*(grid_width/counts[n-2]))):int(((2*i)*((grid_width/counts[n-2]))+(2*(grid_width/counts[n-2]))))])
    # determine line width
    lw = title_axes[list(title_axes.keys())[0]].get_window_extent().height * 0.01

    # combine dictionaries of titles and legends
    key_titles = ['Legend', 'Overlap Only', 'Merge', '      Nth Order Overlap', '      Lower Order Overlap'] + list(y_titles.values())

    #######################
    # Determine Font Size #
    #######################
    width = min([axes[ax].get_window_extent().transformed(fig.dpi_scale_trans.inverted()).width * fig.dpi for ax in axes.keys()])
    height = (((title_axes[orgs].get_window_extent().transformed(fig.dpi_scale_trans.inverted()).height - (lw/36)) * fig.dpi) / 2)
    fs = width
    for title in key_titles:
        f = get_bounded_font_size(width=width, height=height, text=title, base_size=int(fs*2), fig=fig, style=fontstyle)
        if f < fs:
            fs = f

    ########################
    # Cleaning Image Plots #
    ########################
    for org in axes.keys():
        axes[org].spines['top'].set_visible(False)
        axes[org].spines['bottom'].set_visible(False)
        axes[org].spines['right'].set_visible(False)
        axes[org].spines['left'].set_visible(False)
        axes[org].set_xticks([])
        axes[org].set_yticks([])
        axes[org].set_aspect('equal')
        axes[org].imshow(plt_org[org], interpolation='nearest')

    ######################
    # Assigning Y Titles #
    ###################### 
    for title in y_titles.keys():
        axes[title].set_ylabel(y_titles[title], size=fs)

    ######################
    # Assigning X Titles #
    ######################
    for org in title_axes.keys():
        title_axes[org].set_xlim([0,1])
        title_axes[org].set_ylim([0,1])
        if org != "orgs":
            title_axes[org].plot([0.05, 0.95], [0.5, 0.5], color='#000000', lw=(lw), solid_capstyle='round')
            title_axes[org].text(0.5, 0.77, x_titles[org], fontsize=fs, 
                                horizontalalignment='center',
                                verticalalignment='center')
            title_axes[org].text(0.25, 0.23, 'Merge', fontsize=(fs),
                                horizontalalignment='center',
                                verticalalignment='center')
            title_axes[org].text(0.75, 0.23, 'Overlap Only', fontsize=(fs),
                                horizontalalignment='center',
                                verticalalignment='center')
        else:
            title_axes[org].text(0.5, 0.23, x_titles[org], fontsize=fs,
                                 horizontalalignment='center',
                                 verticalalignment='center')
        title_axes[org].spines['top'].set_visible(False)
        title_axes[org].spines['bottom'].set_visible(False)
        title_axes[org].spines['right'].set_visible(False)
        title_axes[org].spines['left'].set_visible(False)
        title_axes[org].set_axis_off()
    
    #####################
    # Setting Up Legend #
    #####################
    axes["key"] = fig.add_subplot(gs[int(grid_height - (len(orgs.split(splitter))*(grid_width/(len(orgs.split(splitter))+1))) + (ti_spec*((len(orgs.split(splitter)))-1))):,
                                     int((2*grid_width)-(grid_width/(len(orgs.split(splitter))+1))):])
    axes["key"].set_xticks([])
    axes["key"].set_yticks([])
    axes["key"].spines['top'].set_visible(False)
    axes["key"].spines['bottom'].set_visible(False)
    axes["key"].spines['right'].set_visible(False)
    axes["key"].spines['left'].set_visible(False)
    legend_ele = [Patch(facecolor=ORG_COLORS[i], edgecolor="#000000", label=org) for i, org in enumerate(orgs.split(splitter))]
    legend_ele += [Patch(facecolor=HO_MERGE, edgecolor="#000000", label="Nth Order Overlap"), 
                   Patch(facecolor=LO_MERGE, edgecolor="#000000", label="Lower Order Overlap")]
    axes["key"].legend(handles=legend_ele, fontsize=((3*fs)//4), loc='right', 
                       frameon=False, mode='expand', title="Legend", title_fontsize=fs)
    print("Done creating figure, please wait for computer to display it.")
    return fig

In [ ]:
plot_n_overlaps('mitoXlysoXperoxXERXLDXgolgi', 'X', organelle_segs, scale, close_viewer=True, view='2D')